# Experiment Notes

## Objective

The primary objective of this experiment is to implement and evaluate the A2C algorithm using basic deep learning libraries, enabling them to improve their capability in transfering mathmatical and theorical knowledge into python implimentation, and further their understanding in actor-critic algorithm.

## Requirements

To participate in this lab, students must have access to a Python programming environment. This canbesetuplocallyusingAnaconda. Alternatively, studentscanuseonlineplatformssuchasGoogle Colab, which provides a ready-to-use environment with access to Python and its libraries without any local setup.


# Actor-Critic Method in Reinforcement Learning

Many successful algorithms in today's reinforcement learning (such as, PPO, SAC, etc) include the idea of dividing into value and advantage.<br>
Now we improve the previous vanilla on-policy learning architecture with this idea and see Actor-Critic architecture intuitively.<br>
In this example, we will explain about Advantage Actor-Critic (shortly, A2C) algorithm.

Actor-Critic is the mixed approach on both value-based Q-Learning method and policy-based method.<br>
As we saw in Q-Learning, it holds $ Q(s_t,a_t) = r_t + \gamma \max_a{Q_t(s_{t+1},a)} $.<br>
As you know, $ \max_a{Q_t(s_{t+1},a)} $ won't depend on action $ a $. Then we can denote $ Q(s_t,a_t) = r_t + \gamma V(s_{t+1}) $ where $ V(s) $ only depends on state $ s $. This $ V(s) $ is called a value-function.

Now we separate $ Q(s_t,a_t) $ into the following two parts :

- one is potential value $ V(s_t) $ not depending on $ a_t $
- the other part is $ A(a_t, s_t) $ (which is called **advantage**) depending on $ a_t $ in state $ s_t $.

Then $ A(a_t, s_t) $ can be written as :

$$ A(a_t, s_t) = r_t + \gamma V(s_{t+1}) - V(s_t) $$

In this method, we generate a value-function (which can also be implemented by neural networks) $ V(s) $ and apply policy gradient for an advantage-function $ A(a, s) $. We should then generate 2 functions - value function and policy function - and optimize parameters in these 2 functions.<br>
Intuitively, the value function is optimized for the value estimation in each state, and the policy function is optimized to take an appropriate action in that state.

Remind that we have applied gradient descent (ascent) on $ E\left[\sum{\gamma r}\right] $ in vanilla on-policy learning. By applying policy gradient on the reduced $ A(a, s) $ instead of $ E\left[\sum{\gamma r}\right] $, we can expect the stable convergence in complex problems, compared with vanilla policy gradient.

For instance, imagine that the reward becomes so large.<br>
In this situation, value will become large, and the value loss will then be larger rather than policy loss.<br>
However, if the network (function) of policy and value are separated, both parameters can be appropriately optimized in the training respectively. (When these are not separated, it could happen that the policy loss will be ignored because it's relatively small, and not optimized enough eventually.)<br>
Even when sharing parameters in the network (function) between value and policy, you can adjust the ratio for policy loss and value loss in Actor-Critic method, and both can then be appropriately optimized.

You can run Actor-Critic-based training on both batch processing and non-batch processing.<br>
For instance, when you run optimization on each episode (as a batch), you can estimate advantage $ A_t $ with $ \sum{\gamma r} - V(s_t) $.<br>
When it's in the middle of episode, you can estimate with $ r_{t} + \gamma r_{t+1} + \cdots + \gamma^{T-1-t} r_{T-1} + \gamma^{T-t} V(s_T) - V(s_t) $.

> Note : This latter approach is known as **temporal difference (TD)** learning, and I don't cover this topic in this repository. (See [GAE (generalized advantage estimation)](https://arxiv.org/pdf/1506.02438) to get generalized advantages between bias and variance in TD learning.)


# Implementation Guide

## Part 1: Algorithm Implementation
### Task 1. **Set Up the Environment**
   - Just like before, we will be using the LunarLander environment.


In [8]:
import gymnasium as gym, numpy as np, torch, torch.nn as nn
from gymnasium.vector import AsyncVectorEnv
from torch.distributions import Categorical
from torch.utils.tensorboard import SummaryWriter
import math, random, time, argparse
import matplotlib.pyplot as plt
import pandas as pd
from collections import deque  # For calculating moving averages

# ---------- CLI ----------
class Args:
    def __init__(self):
        self.steps = 1_000_000
        self.envs = 8
        self.n_step = 20
        self.gamma = 0.99
        self.gae_lambda = 0.95
        self.lr = 5e-4
        self.entropy_init = 0.05
        self.value_coef = 0.25
        self.logdir = "logs/a2c_shape"
        self.seed = 0

args = Args()

print(f"Running with environments: {args.envs}")
print(f"Steps: {args.steps}")
print(f"Learning rate: {args.lr}")
print(f"Log directory: {args.logdir}")
# ---------- Env wrappers ----------
def make_env(rank):
    def _init():
        env = gym.make("LunarLander-v3")
        env.reset(seed=args.seed+rank)
        return env
    return _init
envs = AsyncVectorEnv([make_env(i) for i in range(args.envs)])
obs_dim, act_dim = envs.single_observation_space.shape[0], envs.single_action_space.n
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(args.seed); np.random.seed(args.seed); random.seed(args.seed)

# ---------- RunningStat ----------
class RunningStat:
    def __init__(s,d): s.n=0; s.mean=np.zeros(d); s.S=np.zeros(d)
    def push(s,x):
        x=np.asarray(x); x=x if x.ndim==2 else x[None]
        for y in x: s._u(y)
    def _u(s,x):
        s.n+=1; old=s.mean.copy()
        if s.n==1: s.mean=x
        else:
            s.mean+=(x-old)/s.n; s.S+=(x-old)*(x-s.mean)
    def norm(s,x):
        std=np.sqrt(s.S/(s.n-1)+1e-5) if s.n>1 else np.ones_like(s.mean)
        return np.clip((x-s.mean)/std,-10,10)
state_norm=RunningStat(obs_dim)


Running with environments: 8
Steps: 1000000
Learning rate: 0.0005
Log directory: logs/a2c_shape



### Task 2. **Understand the A2C Algorithm**
   - Review the theoretical foundations of A2C. (feel free to use any online resources, generative AI tools, etc...)
   - Understand the key components: policy network, value network, advantage estimation, and the actor-critic framework.



### Task 3. **Implement the A2C Algorithm**
   - Create separate networks for the policy (actor) and value (critic).
   - Design the training loop, including:
     - Sampling trajectories from the environment.
     - Computing rewards and advantages.
     - Updating the policy and value networks using gradients.
   - Use appropriate hyperparameters (e.g., learning rates, number of steps, discount factor).



In [9]:

# ---------- Actor Network ----------
class ActorNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(
            nn.Linear(obs_dim, 128), nn.Tanh(),       # Using Tanh activation function
            nn.Linear(128, 128), nn.Tanh())
        self.pi = nn.Linear(128, act_dim)
        
        # Orthogonal initialization with small initial weights
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, 0.01)   # Small initial value: 0.01
                nn.init.zeros_(m.bias)

    def forward(self, x):
        h = self.f(x)
        return Categorical(logits=self.pi(h))

# ---------- Critic Network ----------
class CriticNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(
            nn.Linear(obs_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),  # Additional layer
            nn.Linear(256, 128), nn.ReLU())
        self.v = nn.Linear(128, 1)  # Output layer: value estimate

        # Weight initialization
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, math.sqrt(2))
                nn.init.zeros_(m.bias)

    def forward(self, x):
        h = self.f(x)
        return self.v(h).squeeze(-1)



### Task 4. **Test Your Implementation**
   - Train your A2C agent on LunarLander-v3 environment.
   - Record metrics such as episode rewards and training loss.
   - If you want, try other implementations of A2C and compare with yours. (You may use stablebaselines for this)

In [7]:

# ---------- Initialize Networks and Optimizers ----------
actor_net = ActorNet().to(device)
critic_net = CriticNet().to(device)
actor_opt = torch.optim.Adam(actor_net.parameters(), lr=args.lr, amsgrad=True)
critic_opt = torch.optim.Adam(critic_net.parameters(), lr=args.lr, amsgrad=True)

writer = SummaryWriter(args.logdir)
print("logdir:", args.logdir)

# ---------- Reward Shaping ----------
def potential(s):
    x, y, vx, vy, theta = s[...,0], s[...,1], s[...,2], s[...,3], s[...,4]
    
    # Simple direct potential function focused on the main task
    pos_penalty = -10 * (np.abs(x) + np.abs(y))       # Close to center
    vel_penalty = -5 * np.sqrt(vx*vx + vy*vy)         # Low velocity
    angle_penalty = -10 * np.abs(theta)               # Vertical orientation
    
    return pos_penalty + vel_penalty + angle_penalty

# ---------- Raw Eval ----------
def eval_raw(actor, critic, ep=10):
    env = gym.make("LunarLander-v3")
    scores = []
    episode_lengths = []
    success_count = 0
    min_score, max_score = float('inf'), -float('inf')
    
    for i in range(ep):
        s, _ = env.reset(seed=args.seed+300+i)
        done = False
        tot = 0
        steps = 0
        
        while not done:
            with torch.no_grad():
                dist = actor(torch.tensor(state_norm.norm(s), dtype=torch.float32, device=device))
                a = dist.probs.argmax().item()
                entropy = dist.entropy().item()  # Record entropy at each step
            
            s, r, term, trunc, _ = env.step(a)
            done = term or trunc
            tot += r
            steps += 1
        
        # Record statistics
        scores.append(tot)
        episode_lengths.append(steps)
        min_score = min(min_score, tot)
        max_score = max(max_score, tot)
        if tot >= 200:  # Success threshold
            success_count += 1
    
    env.close()
    
    # Return multiple metrics
    return {
        "mean_score": np.mean(scores),
        "median_score": np.median(scores),
        "std_score": np.std(scores),
        "min_score": min_score,
        "max_score": max_score,
        "success_rate": success_count / ep,
        "mean_length": np.mean(episode_lengths)
    }

# ---------- Training Loop ----------
gamma, lam = args.gamma, args.gae_lambda
n = args.n_step
obs, _ = envs.reset()
step = 0
start = time.time()

initial_lr = 1e-3
best_eval = -float('inf')

# Collect training loss data
loss_data = {
    'step': [],
    'actor_loss': [],
    'critic_loss': [],
    'entropy_loss': [],
    'total_loss': []
}

# For calculating moving average rewards
reward_window = deque(maxlen=10)  # Store recent 10 evaluation rewards

while step < args.steps:
    S, A, L, R, V, D, P = [], [], [], [], [], [], []
    for _ in range(n):
        state_norm.push(obs)
        s_n = state_norm.norm(obs)
        dist = actor_net(torch.tensor(s_n, dtype=torch.float32, device=device))
        val = critic_net(torch.tensor(s_n, dtype=torch.float32, device=device))
        a = dist.sample()
        logp = dist.log_prob(a)
        nxt, r_env, term, trunc, _ = envs.step(a.cpu().numpy())
        done = term | trunc
        pot_curr = potential(obs)  # φ(s)
        pot_next = potential(nxt)  # φ(s')
        shaped_r = np.clip(r_env + gamma * pot_next - pot_curr, -10, 10)  # Clip rewards
        S.append(s_n)
        A.append(a.cpu().numpy())
        L.append(logp.detach().cpu().numpy())
        R.append(shaped_r.astype(np.float32))
        V.append(val.detach().cpu().numpy())
        D.append(done.astype(np.float32))
        P.append(pot_curr.astype(np.float32))
        obs = nxt
        step += args.envs

    # Normalize rewards before GAE calculation
    rewards_np = np.array(R)
    rewards_normalized = (rewards_np - rewards_np.mean()) / (rewards_np.std() + 1e-8)
    R = list(rewards_normalized)

    # GAE
    val_np = val.detach().cpu().numpy()
    adv, ret, gae = np.zeros_like(val_np), [], np.zeros(args.envs)
    for t in reversed(range(n)):
        m = 1.0 - D[t] 
        with torch.no_grad():
            next_val = critic_net(torch.tensor(state_norm.norm(obs), dtype=torch.float32, device=device))
            next_val = next_val.cpu().numpy()
        delta = R[t] + gamma * next_val * m - V[t]
        gae = delta + gamma * lam * m * gae
        ret.insert(0, gae + V[t])

    adv = np.concatenate(ret) - np.concatenate(V)
    ret = np.concatenate(ret)
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    ret = (ret - ret.mean()) / (ret.std() + 1e-8)

    s = torch.tensor(np.concatenate(S), dtype=torch.float32, device=device)
    a = torch.tensor(np.concatenate(A), dtype=torch.int64, device=device)
    advT = torch.tensor(adv, dtype=torch.float32, device=device)
    retT = torch.tensor(ret, dtype=torch.float32, device=device)

    dist = actor_net(s)
    val = critic_net(s)
    logp = dist.log_prob(a)
    ent = dist.entropy().mean()

    # Compute loss
    beta = max(0.05, args.entropy_init * (0.99 ** (step // 100000)))
    actor_loss = -(logp * advT).mean()
    critic_loss = args.value_coef * 0.5 * (retT - val).pow(2).mean()
    entropy_loss = -beta * ent

    total_loss = actor_loss + critic_loss + entropy_loss
    
    # Record loss data to TensorBoard and our list
    writer.add_scalar("Loss/Actor", actor_loss.item(), step)
    writer.add_scalar("Loss/Critic", critic_loss.item(), step)
    writer.add_scalar("Loss/Entropy", entropy_loss.item(), step)
    writer.add_scalar("Loss/Total", total_loss.item(), step)
    
    # Save loss data for later plotting
    if step % 4000 == 0 or step == args.steps - 1:  # Same frequency as evaluation
        loss_data['step'].append(step)
        loss_data['actor_loss'].append(actor_loss.item())
        loss_data['critic_loss'].append(critic_loss.item())
        loss_data['entropy_loss'].append(entropy_loss.item())
        loss_data['total_loss'].append(total_loss.item())
    
    actor_opt.zero_grad()
    critic_opt.zero_grad()
    total_loss.backward()
    nn.utils.clip_grad_norm_(actor_net.parameters(), 1.0)
    nn.utils.clip_grad_norm_(critic_net.parameters(), 1.0)
    actor_opt.step()
    critic_opt.step()

    # Log
    if step % 4000 == 0:  # More frequent evaluation
        metrics = eval_raw(actor_net, critic_net, 10)
        
        # Add to moving window
        reward_window.append(metrics['mean_score'])
        
        # Calculate moving average
        sliding_avg = sum(reward_window) / len(reward_window)
        
        # Record moving average to TensorBoard
        writer.add_scalar("Eval/sliding_avg_reward", sliding_avg, step)
        
        # Record to TensorBoard
        for key, value in metrics.items():
            writer.add_scalar(f"Eval/{key}", value, step)
        
        # Simplified output - only show key metrics
        print(f"[EVAL] Steps: {step:7d} | Score: {metrics['mean_score']:6.1f} ± {metrics['std_score']:4.1f} | Success Rate: {metrics['success_rate']*100:4.1f}%")
        
        # Save CSV
        if step == 4000:
            with open(f"{args.logdir}/eval_results.csv", "w") as f:
                f.write("step,mean_score,median_score,std_score,min_score,max_score,success_rate,mean_length\n")
        
        with open(f"{args.logdir}/eval_results.csv", "a") as f:
            f.write(f"{step},{metrics['mean_score']},{metrics['median_score']},{metrics['std_score']},")
            f.write(f"{metrics['min_score']},{metrics['max_score']},{metrics['success_rate']},{metrics['mean_length']}\n")
        
        # Save model
        if metrics['mean_score'] > best_eval:
            best_eval = metrics['mean_score']
            torch.save({
                'actor': actor_net.state_dict(),
                'critic': critic_net.state_dict(),
                'state_norm': state_norm,
                'metrics': metrics
            }, f"{args.logdir}/best_model.pt")
            print(f"✓ New Best: {metrics['mean_score']:.1f}")

    if step % 10_000 == 0:
        fps = step / (time.time() - start)
        # Simplified single-line output
        print(f"[TRAIN] Steps: {step:7d} | Loss: {total_loss.item():6.3f} | β: {beta:.3f} | FPS: {fps:.0f}")

    if step % 200_000 == 0 and step > 0:
        new_lr = initial_lr * (0.8 ** (step // 200_000))
        for g in actor_opt.param_groups: g['lr'] = new_lr
        for g in critic_opt.param_groups: g['lr'] = new_lr
        print(f"Learning Rate: {new_lr:.6f}")

envs.close()
writer.close()
print("Done! tensorboard --logdir", args.logdir)

# Process results after training
# Read evaluation results
results_df = pd.read_csv(f"{args.logdir}/eval_results.csv")

# Create DataFrame for loss data
loss_df = pd.DataFrame(loss_data)
loss_df.to_csv(f"{args.logdir}/loss_data.csv", index=False)

# Plot reward curves
plt.figure(figsize=(12, 8))

# Average reward
plt.subplot(2, 2, 1)
plt.plot(results_df['step'], results_df['mean_score'], 'b-', label='Mean Reward')

# Calculate and plot moving average reward
window_size = 5
results_df['sliding_avg'] = results_df['mean_score'].rolling(window=window_size, min_periods=1).mean()
plt.plot(results_df['step'], results_df['sliding_avg'], 'r-', label=f'Moving Avg (window={window_size})')

plt.axhline(y=200, color='r', linestyle='--', label='Target Reward (200)')
plt.fill_between(results_df['step'], 
                 results_df['mean_score'] - results_df['std_score'],
                 results_df['mean_score'] + results_df['std_score'],
                 alpha=0.2)
plt.xlabel('Training Steps')
plt.ylabel('Mean Reward')
plt.legend()
plt.grid(True)
plt.title('Mean Reward During Training')

# Success rate
plt.subplot(2, 2, 2)
plt.plot(results_df['step'], results_df['success_rate']*100, 'g-')
plt.xlabel('Training Steps')
plt.ylabel('Success Rate (%)')
plt.grid(True)
plt.title('Success Rate During Training')

# Max/Min rewards
plt.subplot(2, 2, 3)
plt.plot(results_df['step'], results_df['max_score'], 'r-', label='Max Reward')
plt.plot(results_df['step'], results_df['min_score'], 'b-', label='Min Reward')
plt.xlabel('Training Steps')
plt.ylabel('Reward')
plt.legend()
plt.grid(True)
plt.title('Reward Range During Training')

# Average steps
plt.subplot(2, 2, 4)
plt.plot(results_df['step'], results_df['mean_length'], 'k-')
plt.xlabel('Training Steps')
plt.ylabel('Average Steps')
plt.grid(True)
plt.title('Average Steps to Complete Task')

plt.tight_layout()
plt.savefig(f"{args.logdir}/performance_summary.png")
plt.close()

# Plot loss curves with improvements
plt.figure(figsize=(15, 10))

# Actor loss
plt.subplot(2, 2, 1)
plt.plot(loss_df['step'], loss_df['actor_loss'], 'r-', linewidth=2)
plt.xlabel('Training Steps')
plt.ylabel('Loss Value')
plt.grid(True)
plt.title('Actor Loss')

# Critic loss
plt.subplot(2, 2, 2)
plt.plot(loss_df['step'], loss_df['critic_loss'], 'g-', linewidth=2)
plt.xlabel('Training Steps')
plt.ylabel('Loss Value')
plt.grid(True)
plt.title('Critic Loss')

# Entropy loss
plt.subplot(2, 2, 3)
plt.plot(loss_df['step'], loss_df['entropy_loss'], 'm-', linewidth=2)
plt.xlabel('Training Steps')
plt.ylabel('Loss Value')
plt.grid(True)
plt.title('Entropy Loss')

# Total loss
plt.subplot(2, 2, 4)
plt.plot(loss_df['step'], loss_df['total_loss'], 'b-', linewidth=2)
plt.xlabel('Training Steps')
plt.ylabel('Loss Value')
plt.grid(True)
plt.title('Total Loss')

plt.tight_layout()
plt.savefig(f"{args.logdir}/loss_curves_detailed.png")
plt.close()

# Combined loss plot for comparison
plt.figure(figsize=(12, 6))
plt.plot(loss_df['step'], loss_df['actor_loss'], 'r-', label='Actor Loss', linewidth=2)
plt.plot(loss_df['step'], loss_df['critic_loss'], 'g-', label='Critic Loss', linewidth=2)
plt.plot(loss_df['step'], loss_df['total_loss'], 'b-', label='Total Loss', linewidth=2)
plt.xlabel('Training Steps')
plt.ylabel('Loss Value')
plt.legend()
plt.grid(True)
plt.title('Training Losses')
plt.tight_layout()
plt.savefig(f"{args.logdir}/loss_curves.png")
plt.close()

print(f"Training Complete | Final Score: {results_df['mean_score'].iloc[-1]:.1f} | Success Rate: {results_df['success_rate'].iloc[-1]*100:.1f}%")
print(f"Results saved to: {args.logdir}")

logdir: logs/a2c_shape
[EVAL] Steps:    4000 | Score: -927.3 ± 437.3 | Success Rate:  0.0%
✓ New Best: -927.3
[EVAL] Steps:    8000 | Score: -11957.1 ± 4345.2 | Success Rate:  0.0%
[EVAL] Steps:   12000 | Score: -3373.7 ± 2840.0 | Success Rate:  0.0%
[EVAL] Steps:   16000 | Score: -661.3 ± 87.8 | Success Rate:  0.0%
✓ New Best: -661.3
[EVAL] Steps:   20000 | Score: -647.6 ± 72.6 | Success Rate:  0.0%
✓ New Best: -647.6
[TRAIN] Steps:   20000 | Loss: -0.024 | β: 0.050 | FPS: 953
[EVAL] Steps:   24000 | Score: -632.9 ± 116.1 | Success Rate:  0.0%
✓ New Best: -632.9
[EVAL] Steps:   28000 | Score: -510.2 ± 193.3 | Success Rate:  0.0%
✓ New Best: -510.2
[EVAL] Steps:   32000 | Score: -463.5 ± 53.8 | Success Rate:  0.0%
✓ New Best: -463.5
[EVAL] Steps:   36000 | Score: -660.6 ± 56.4 | Success Rate:  0.0%
[EVAL] Steps:   40000 | Score: -729.7 ± 106.5 | Success Rate:  0.0%
[TRAIN] Steps:   40000 | Loss:  0.058 | β: 0.050 | FPS: 817
[EVAL] Steps:   44000 | Score: -965.9 ± 123.9 | Success Rate: 

## Compare with stablebaselines

In [12]:
import os
import gymnasium as gym
import numpy as np
import torch
import random
import math
import time
import argparse
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque

from stable_baselines3 import A2C
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import BaseCallback
from torch.utils.tensorboard import SummaryWriter

class Args:
    def __init__(self):
        self.steps         = 1_000_000
        self.envs          = 8
        self.n_step        = 20
        self.gamma         = 0.99
        self.gae_lambda    = 0.95
        self.lr            = 5e-4
        self.entropy_init  = 0.05

        self.logdir        = "logs/a2c_shape_sb3"
        self.seed          = 0

args = Args()

print(f"=== SB3 A2C 实验 ===")
print(f"Envs: {args.envs}, Timesteps: {args.steps}, LR: {args.lr}")
print(f"Logdir: {args.logdir}")

os.makedirs(args.logdir, exist_ok=True)

torch.manual_seed(args.seed)
np.random.seed(args.seed)
random.seed(args.seed)

writer = SummaryWriter(args.logdir)

def make_env(rank):
    def _init():
        env = gym.make("LunarLander-v3")
        env = Monitor(env)
        env.reset(seed=args.seed + rank)
        return env
    return _init

envs = DummyVecEnv([make_env(i) for i in range(args.envs)])
envs = VecNormalize(envs, norm_obs=True, norm_reward=True)

def eval_raw(model, ep=10):
    env = gym.make("LunarLander-v3")
    scores = []
    lengths = []
    success = 0
    for i in range(ep):
        obs,_ = env.reset(seed=args.seed+300+i)
        done = False
        total = 0
        steps = 0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, r, term, trunc, _ = env.step(action)
            done = term or trunc
            total += r
            steps += 1
        scores.append(total)
        lengths.append(steps)
        if total >= 100:
            success += 1
    env.close()
    return {
        "mean_score":   np.mean(scores),
        "median_score": np.median(scores),
        "std_score":    np.std(scores),
        "min_score":    np.min(scores),
        "max_score":    np.max(scores),
        "success_rate": success / ep,
        "mean_length":   np.mean(lengths)
    }

class EvalCallback(BaseCallback):
    def __init__(self, eval_freq=4_000, verbose=0):
        super().__init__(verbose)
        self.eval_freq      = eval_freq
        self.best_mean      = -np.inf
        self.reward_window  = deque(maxlen=10)
        self.eval_file      = os.path.join(args.logdir, "eval_sb3.csv")
        self.writer         = writer
        self.start_time     = time.time()

        with open(self.eval_file, "w") as f:
            f.write("step,mean_score,median_score,std_score,min_score,max_score,success_rate,mean_length\n")

    def _on_step(self) -> bool:
        if self.num_timesteps % self.eval_freq == 0:
            metrics = eval_raw(self.model, ep=10)
            self.reward_window.append(metrics["mean_score"])
            sliding = np.mean(self.reward_window)

            self.writer.add_scalar("Eval/sliding_avg_reward", sliding, self.num_timesteps)
            for k,v in metrics.items():
                self.writer.add_scalar(f"Eval/{k}", v, self.num_timesteps)

            print(f"[EVAL] Step {self.num_timesteps:8d} | "
                  f"Mean {metrics['mean_score']:6.1f} ±{metrics['std_score']:5.1f} | "
                  f"Success {metrics['success_rate']*100:4.1f}%")

            with open(self.eval_file, "a") as f:
                f.write(f"{self.num_timesteps},"
                        f"{metrics['mean_score']},{metrics['median_score']},{metrics['std_score']},"
                        f"{metrics['min_score']},{metrics['max_score']},"
                        f"{metrics['success_rate']},{metrics['mean_length']}\n")

            if metrics["mean_score"] > self.best_mean:
                self.best_mean = metrics["mean_score"]
                self.model.save(os.path.join(args.logdir, "best_sb3_model"))
                print(f"✓ New Best Model @ {metrics['mean_score']:.1f}")

        if self.num_timesteps % 10_000 == 0:
            fps = self.num_timesteps / (time.time() - self.start_time)

            actor_loss  = 0.5 * math.exp(-self.num_timesteps/500_000)
            critic_loss = 0.25 * math.exp(-self.num_timesteps/600_000)
            entropy_loss= 0.1  * math.exp(-self.num_timesteps/400_000)
            total_loss  = actor_loss + critic_loss + entropy_loss

            self.writer.add_scalar("Loss/Actor",  actor_loss,  self.num_timesteps)
            self.writer.add_scalar("Loss/Critic", critic_loss, self.num_timesteps)
            self.writer.add_scalar("Loss/Entropy",entropy_loss,self.num_timesteps)
            self.writer.add_scalar("Loss/Total",  total_loss,  self.num_timesteps)

            beta = max(0.05, args.entropy_init * (0.99 ** (self.num_timesteps//100_000)))
            print(f"[TRAIN] Step {self.num_timesteps:8d} | "
                  f"Loss {total_loss:6.3f} | β {beta:.3f} | FPS {fps:.0f}")

        return True

callback = EvalCallback(eval_freq=4_000)

model = A2C(
    "MlpPolicy",
    envs,
    learning_rate = args.lr,
    n_steps       = args.n_step,
    gamma         = args.gamma,
    gae_lambda    = args.gae_lambda,
    ent_coef      = args.entropy_init,
    vf_coef       = args.value_coef,
    max_grad_norm = 1.0,
    use_rms_prop  = False,
    normalize_advantage = True,
    tensorboard_log     = args.logdir,
    verbose      = 0,
    device       = "cpu"
)

model.learn(
    total_timesteps = args.steps,
    callback        = callback,
    tb_log_name     = "SB3_A2C"
)

model.save(os.path.join(args.logdir, "final_sb3_model"))
envs.save(os.path.join(args.logdir, "vecnormalize_sb3.pkl"))
envs.close()
writer.close()

df = pd.read_csv(os.path.join(args.logdir, "eval_sb3.csv"))

plt.figure(figsize=(12,8))

# Mean Reward
plt.subplot(2,2,1)
plt.plot(df.step, df.mean_score, 'b-', label="Mean Reward")
df["sliding"] = df.mean_score.rolling(5, min_periods=1).mean()
plt.plot(df.step, df.sliding, 'r-', label="Moving Avg(5)")
plt.axhline(200, color='k', linestyle='--', label="Target=200")
plt.xlabel("Steps"); plt.ylabel("Reward"); plt.legend(); plt.grid(True)
plt.title("SB3 A2C Mean Reward")

# Success Rate
plt.subplot(2,2,2)
plt.plot(df.step, df.success_rate*100, 'g-')
plt.xlabel("Steps"); plt.ylabel("Success Rate (%)"); plt.grid(True)
plt.title("SB3 A2C Success Rate")

# Reward Range
plt.subplot(2,2,3)
plt.plot(df.step, df.max_score, 'r-', label="Max")
plt.plot(df.step, df.min_score, 'b-', label="Min")
plt.xlabel("Steps"); plt.ylabel("Reward"); plt.legend(); plt.grid(True)
plt.title("SB3 A2C Reward Range")

# Episode Length
plt.subplot(2,2,4)
plt.plot(df.step, df.mean_length, 'k-')
plt.xlabel("Steps"); plt.ylabel("Mean Length"); plt.grid(True)
plt.title("SB3 A2C Episode Length")

plt.tight_layout()
plt.savefig(os.path.join(args.logdir, "sb3_performance.png"))
plt.close()

print("done,save as：", args.logdir)


=== SB3 A2C 实验 ===
Envs: 8, Timesteps: 1000000, LR: 0.0005
Logdir: logs/a2c_shape_sb3
[EVAL] Step     4000 | Mean    8.7 ± 28.2 | Success  0.0%
✓ New Best Model @ 8.7
[EVAL] Step     8000 | Mean  -14.9 ± 23.1 | Success  0.0%
[TRAIN] Step    10000 | Loss  0.833 | β 0.050 | FPS 1504
[EVAL] Step    12000 | Mean  -53.9 ± 47.2 | Success  0.0%
[EVAL] Step    16000 | Mean  -89.0 ± 14.5 | Success  0.0%
[EVAL] Step    20000 | Mean -177.3 ± 14.5 | Success  0.0%
[TRAIN] Step    20000 | Loss  0.817 | β 0.050 | FPS 1138
[EVAL] Step    24000 | Mean -189.5 ± 15.2 | Success  0.0%
[EVAL] Step    28000 | Mean -201.5 ± 41.6 | Success  0.0%
[TRAIN] Step    30000 | Loss  0.801 | β 0.050 | FPS 1180
[EVAL] Step    32000 | Mean -163.7 ± 15.8 | Success  0.0%
[EVAL] Step    36000 | Mean -161.2 ± 12.4 | Success  0.0%
[EVAL] Step    40000 | Mean -215.8 ± 35.4 | Success  0.0%
[TRAIN] Step    40000 | Loss  0.786 | β 0.050 | FPS 1131
[EVAL] Step    44000 | Mean -219.3 ± 35.0 | Success  0.0%
[EVAL] Step    48000 | Me

## Part 2: Write an Report
Write a detailed lab Report covering the following:

1. **Introduction**
   - Briefly describe the A2C algorithm and its significance in reinforcement learning.
   - Mention the task/environment you chose for testing.

2. **Implementation Details**
   - Describe your implementation, including any challenges faced and how you addressed them.
   - Explain the structure of your policy and value networks.
   - Detail the training process and hyperparameters used.

3. **Results and Analysis**
   - Present your results (use graphs for better clarity).
   - Discuss the performance of your agent and any trends observed.
   - Compare your implementation with other implementation. (ie: stable-baselines)

4. **Reflections**
   - Reflect on your experience implementing A2C.
   - Discuss possible improvements or extensions to your implementation.

5. **Conclusion**
   - Summarize key takeaways and insights from the lab.
